### ЗАДАЧА: Бронирование переговорок

Офис-менеджер получает пачку заявок на переговорки.
Нужно собрать систему, которая:
- принимает корректные бронирования,
- отбрасывает конфликтующие или неправильные заявки,
- хранит расписание по комнатам,
- позволяет быстро понять, какая переговорка загружена сильнее всего.

В некоторых заявках указана неизвестная комната,
в некоторых время перепутано,
а некоторые пересекаются с уже занятыми слотами.


In [24]:
from dataclasses import dataclass
from typing import Optional


rooms = {'A-101', 'B-204', 'C-305'}
# rows: booking_id|room_id|owner|start_hour|end_hour
rows = [
    'BK-100|A-101|Alice|9|11',
    'BK-101|A-101|Bob|10|12',
    'BK-102|B-204|Kira|13|15',
    'BK-103|X-999|Oleg|11|12',
    'BK-104|C-305|Eva|16|15',
    'BK-105|B-204|Max|15|17',
]


class BookingError(Exception):
    pass
 

class RoomNotFoundError(BookingError):
    pass


class TimeRangeError(BookingError):
    pass


class TimeConflictError(BookingError):
    pass


@dataclass(order=True)
class Booking:
    start_hour: int
    end_hour: int
    booking_id: str
    room_id: str
    owner: str


class RoomSchedule:
    def __init__(self, room_id):
        # TODO: сохранить room_id
        # TODO: создать список bookings
        self.room_id = room_id
        self.bookings = []

    def can_add(self, booking):
        # TODO: пройтись по уже существующим booking в self.bookings
        # TODO: проверить пересечение интервалов
        # TODO: если пересечение есть -> вернуть False
        # TODO: если конфликтов нет -> вернуть True
        for el in self.bookings:
            if el.end_hour > booking.start_hour:
                return False
        return True


    def add_booking(self, booking):
        # TODO: если can_add(...) == False -> raise TimeConflictError(...)
        # TODO: добавить booking в self.bookings
        # TODO: отсортировать self.bookings
        if not self.can_add(booking):
            raise TimeConflictError("некорректное время!")
        else:
            self.bookings.append(booking)
            self.bookings.sort()

    def total_reserved_hours(self):
        # TODO: вернуть сумму (end_hour - start_hour) по всем бронированиям комнаты
        start = 0
        end = 0
        for el in self.bookings:
            arr = el.split("|")
            start += int(arr[3])
            end += int(arr[4])
        return end - start
        


class BookingService:
    def __init__(self, rooms):
        # TODO: создать schedules вида room_id -> RoomSchedule(room_id)
        # TODO: создать списки accepted и errors
        self.schedules = { }
        self.accepted = []
        self.errors = []
    def parse_booking(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: booking_id, room_id, owner, start_raw, end_raw
        # TODO: start_raw и end_raw преобразовать в int
        # TODO: если room_id не существует -> RoomNotFoundError
        # TODO: если start_hour >= end_hour -> TimeRangeError
        # TODO: вернуть объект Booking(...)
        booking_id, room_id, owner, start_raw, end_raw = row.split('|')
        start_raw, end_raw = int(start_raw), int(end_raw)
        if room_id not in rooms:
            raise RoomNotFoundError('такой комнаты нет!')
        if start_raw >= end_raw:
            raise TimeRangeError('время начала бронирования больше времени конца!')
        
        return Booking(start_hour=start_raw, end_hour=end_raw, booking_id=booking_id, room_id=room_id, owner=owner)

    def submit(self, row):
        # TODO: внутри try вызвать parse_booking(row)
        # TODO: затем schedules[booking.room_id].add_booking(booking)
        # TODO: успех добавить в self.accepted
        # TODO: BookingError сохранить в self.errors как (row, error_type, message)
        try:
            num = row.split('|')[1]
            b = self.parse_booking(row)
            self.schedules.setdefault(b.room_id, [])
            self.schedules[b.room_id].append(RoomSchedule(num))
            for el in self.schedules[b.room_id]:
                el.add_booking(b)
            self.accepted.append(b)
        except Exception as e:
            self.errors.append((row, type(e).__name__, e))



    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for el in rows:
            self.submit(el)

    def busiest_room(self):
        # TODO: найти комнату с максимумом total_reserved_hours()
        # TODO: вернуть tuple(room_id, total_hours)
        m = 0
        max_hours_room = None
        for el in self.accepted:
            diff = (el.end_hour - el.start_hour)
            if diff > m:
                m = diff
                max_hours_room = el.room_id
        return (m, max_hours_room)


        

    def find_booking(self, booking_id):
        # TODO: вернуть Optional[Booking]
        # TODO: пройтись по всем schedules и по всем bookings внутри них
        # TODO: если booking.booking_id совпал -> вернуть booking
        # TODO: если не найдено -> вернуть None
        for key, val in self.schedules.items():
            for l in val:
                for el in l.bookings:
                    if el.booking_id == booking_id:
                        return el
        return None


service = BookingService(rooms)

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые бронирования
# TODO: вывести ошибки
# TODO: вывести расписание по всем комнатам
# TODO: вывести busiest_room()
# TODO: вывести find_booking('BK-102')
service.load(rows=rows)

print('Принятые бронирования: ', service.accepted)
print('Ошибки: ', service.errors)
for key, val in service.schedules.items():
    for i in val:
        
        print(f'{key} - {i.bookings}')
print(service.busiest_room())
print(service.find_booking('BK-102'))

Принятые бронирования:  [Booking(start_hour=9, end_hour=11, booking_id='BK-100', room_id='A-101', owner='Alice'), Booking(start_hour=13, end_hour=15, booking_id='BK-102', room_id='B-204', owner='Kira'), Booking(start_hour=15, end_hour=17, booking_id='BK-105', room_id='B-204', owner='Max')]
Ошибки:  [('BK-101|A-101|Bob|10|12', 'TimeConflictError', TimeConflictError('некорректное время!')), ('BK-103|X-999|Oleg|11|12', 'RoomNotFoundError', RoomNotFoundError('такой комнаты нет!')), ('BK-104|C-305|Eva|16|15', 'TimeRangeError', TimeRangeError('время начала бронирования больше времени конца!'))]
A-101 - [Booking(start_hour=9, end_hour=11, booking_id='BK-100', room_id='A-101', owner='Alice')]
A-101 - []
B-204 - [Booking(start_hour=13, end_hour=15, booking_id='BK-102', room_id='B-204', owner='Kira'), Booking(start_hour=15, end_hour=17, booking_id='BK-105', room_id='B-204', owner='Max')]
B-204 - [Booking(start_hour=15, end_hour=17, booking_id='BK-105', room_id='B-204', owner='Max')]
(2, 'A-101')